In [1]:
import torch
import einops
import random

In [2]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [3]:
def dag_loss(targets, transition_matrix, emission_probs, bos_idx=0):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.ones((batch_size, m, l))
    dp[dp == 1] = -float('inf')
    initial_probs = torch.gather(emission_probs, dim=2, index=targets[:, 0].unsqueeze(1).unsqueeze(2))
    dp[:, 0, 0] = initial_probs.squeeze(2).squeeze(1)
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (torch.logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [4]:
def process_dp(dp, target_lens, vertex_lens):
    dp_values = vector_gather(dp, target_lens - 1)
    values = torch.gather(dp_values, dim=1, index=(vertex_lens - 1).unsqueeze(-1))
    return values

In [5]:
def fix_probs(logprobs, mask):
    # assumes probs is already in log space
    # and is a square matrix
    # updates probs so that the sum of each row is 1
    # and any available probability mass is 
    # distributed evenly among the non-masked entries
    batch_size, l, _ = logprobs.shape

    # any part of the mask where the value is not 0, we mask out
    logprobs = logprobs.masked_fill(mask != 0, float('-inf'))
    probsmatrix = torch.exp(logprobs)
    remaining = torch.sum(probsmatrix, dim=2)
    remaining = 1 - remaining
    probnonzero = torch.sum(mask == 0, dim=-1)
    remaining = remaining / probnonzero
    probsmatrix = probsmatrix + remaining.unsqueeze(2)

    # at this point, it is mostly correct, however if there 
    # are rows where the number of non zeros (probnonzero)
    # is 0, then we get infinities at those positions, which
    # are obviously wrong, so we keep only positions
    # where we haven't masked out
    probsmatrix = probsmatrix.masked_fill(mask != 0, 0)
    logprobs = torch.log(probsmatrix)
    return logprobs

In [6]:
def acyclic_mask(transition_matrix):
    batch_size, vertices, _ = transition_matrix.shape
    mask = torch.tril(torch.ones((vertices, vertices)))
    return mask

In [7]:
def padding_transition_mask(transition_matrix, vertex_lens):
    """
    vertex_lens is a tensor of shape (batch_size,)
    which describes which vertices for each batch are not padding
    """
    batch_size, vertices, _ = transition_matrix.shape
    vertex_lens_mask = torch.arange(vertices).repeat(len(vertex_lens), 1) < vertex_lens.unsqueeze(-1)
    mask = torch.ones_like(transition_matrix)
    mask.transpose(1,2)[vertex_lens_mask] = 0
    return mask

In [8]:
def masking(transition_matrix, vertex_lens):
    acyclic = acyclic_mask(transition_matrix)
    padding = padding_transition_mask(transition_matrix, vertex_lens)
    return padding + acyclic

In [9]:
vocab_size = 5
hidden_dim = 10
batch_size = 2
factor = 2

In [10]:
pad_to_len = 8

In [11]:
target_tensors = []
target_tensor_lengths = []
for _ in range(batch_size):
    # based on tests, the loss function doesn't work if the target length
    # is less than or equal to 1, so we do 2 up to pad_to_len - 1 instead
    # see note at bottom of notebook
    target_len = random.randint(2, pad_to_len - 1)
    target_tensor_lengths.append(target_len)
    target = torch.randint(low=0, high=vocab_size, size=(target_len,))
    target_tensors.append(target)


In [12]:
target_tensors, target_tensor_lengths

([tensor([2, 3, 4, 0, 3]), tensor([0, 4])], [5, 2])

In [13]:
target_lengths = torch.tensor(target_tensor_lengths)

In [14]:
vertex_lens = target_lengths * factor

In [15]:
target_tensors = torch.nested.nested_tensor(target_tensors)

/tmp/ipykernel_30356/670723154.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  target_tensors = torch.nested.nested_tensor(target_tensors)


In [16]:
target_tensors = torch.nested.to_padded_tensor(target_tensors, 0, (batch_size, pad_to_len))

In [17]:
target_tensors

tensor([[2, 3, 4, 0, 3, 0, 0, 0],
        [0, 4, 0, 0, 0, 0, 0, 0]])

In [18]:
class TestModel(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim, factor, pad_to_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.factor = factor
        self.pad_to_len = pad_to_len
        self.positional_embedding = torch.nn.Embedding(pad_to_len * factor, hidden_dim)
        self.layer1 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.token_prob_layer = torch.nn.Linear(hidden_dim, vocab_size)
        self.transition_query_layer = torch.nn.Linear(hidden_dim, hidden_dim)
        self.transition_key_layer = torch.nn.Linear(hidden_dim, hidden_dim)

    def forward(self, target_tensors):
        # in the real world we would just need batch_size and length,
        # not the target_tensors (because obviously we don't have the targets during inference)
        batch_size, length = target_tensors.shape
        emb = self.positional_embedding(torch.arange(length * self.factor).unsqueeze(0).expand(batch_size, -1))
        layer1_out = self.layer1(emb)
        # and then send it through activation
        layer1_out = torch.nn.functional.relu(layer1_out)
        token_logits = self.token_prob_layer(layer1_out)
        tq = self.transition_query_layer(layer1_out)
        tk = self.transition_key_layer(layer1_out)
        transition_matrix = tq @ tk.transpose(1, 2)
        token_log_probs = torch.nn.functional.log_softmax(token_logits, dim=-1)
        transition_log_probs = torch.nn.functional.log_softmax(transition_matrix, dim=-1)
        return token_log_probs, transition_log_probs

In [19]:
model = TestModel(vocab_size, hidden_dim, factor, pad_to_len)

In [20]:
from torch.optim import Adam

In [21]:
optm = Adam(model.parameters(), lr=0.001)

In [22]:
token_log_probs, transition_matrix = model(target_tensors)

In [23]:
transition_mask = masking(transition_matrix, vertex_lens)

In [24]:
transition_log_probs = fix_probs(transition_matrix, transition_mask)

In [25]:
dp = dag_loss(target_tensors, transition_log_probs, token_log_probs)

In [26]:
loss_values = process_dp(dp, target_lengths, vertex_lens)

In [27]:
# we want sum of negative log 
loss = -torch.sum(loss_values)

In [28]:
loss

tensor(14.2575, grad_fn=<NegBackward0>)

In [29]:
loss.backward()

/run/media/john/Secondary/Projects/ML/LMTests/lmtest/lib64/python3.11/site-packages/torch/autograd/__init__.py:251: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [30]:
optm.step()

Loss function doesn't work when target length is less than or equal to 1, however this is not an issue with how the dynamic programming approach works, just the way the loss function is formulated. The loss function consideres the sum of probabilities of all paths starting at vertex 1 and ending at the final vertex (let's assume no padding for simplicity). The number of nodes in the path must be exactly equal to the number of tokens in the target sequence, and if the target sequence of length 1, there are no paths and so the probability is 0 and always will be. This is because we must have at least 2 nodes in the path, one for the start and one for the end. So don't try to use this with target sequences of length 1 or less.